# Elliptic++: blind-zone analysis and expert retraining

Run the cells in order (Google Colab, CPU). The Elliptic++ files (`txs_classes.csv`, `txs_features.csv`, `AddrTx_edgelist.txt`, `TxAddr_edgelist.txt`) go in `ROOT/elliptic_plus_plus/`.

## Helpers (LightGBM expert, metrics, saving)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

SEEDS = [42, 123, 2024, 7, 101, 202]
LGBM = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_child_samples=20, feature_fraction=0.8,
            bagging_fraction=0.8, bagging_freq=1, reg_lambda=1.0, n_estimators=3000, n_jobs=-1, verbose=-1)


def fit(Xtr, ytr, Xva, yva, seed):
    import lightgbm as lgb
    m = lgb.LGBMClassifier(random_state=seed, **LGBM)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="average_precision", callbacks=[lgb.early_stopping(150, verbose=False)])
    return m


def metrics(y, s):
    if not 0 < y.sum() < len(y):
        return dict(AUC_ROC=np.nan, AUC_PR=np.nan, lift=np.nan, P_at_R50=np.nan)
    ap = float(average_precision_score(y, s)); p, rc, _ = precision_recall_curve(y, s)
    return dict(AUC_ROC=float(roc_auc_score(y, s)), AUC_PR=ap, lift=ap / y.mean(), P_at_R50=float(p[rc >= 0.5].max()))


class Boot:
    def __init__(self, y, s):
        self.o = np.argsort(-s, kind="stable"); ss = s[self.o]; self.y = y[self.o].astype(np.int64)
        self.st = np.r_[0, np.flatnonzero(np.diff(ss) != 0) + 1]
    def metrics(self, w):
        w = w[self.o]; tp = np.add.reduceat(w * self.y, self.st); fp = np.add.reduceat(w * (1 - self.y), self.st)
        TP = np.cumsum(tp); FP = np.cumsum(fp)
        if TP[-1] == 0 or FP[-1] == 0:
            return float("nan"), float("nan")
        P = TP / np.maximum(TP + FP, 1); R = TP / TP[-1]; ap = float(np.sum(np.diff(R, prepend=0.0) * P))
        x = np.r_[0.0, FP / FP[-1]]; yv = np.r_[0.0, R]
        return ap, float(np.sum(np.diff(x) * (yv[1:] + yv[:-1]) / 2.0))


def save(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        json.dump(obj, open(path, "w"), indent=1)

print("outils prets")

## Elliptic++ loading and blind-zone analysis

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

T_TR, T_VA, T_TEST0, T_END = 24, 34, 35, 49
ELL_POL = ("static", "win3", "all")
ELL_PAIRS = [("rule_win3", "rule_static"), ("rule_all", "rule_static"), ("rule_win3", "rule_all"), ("strict_win3", "rule_win3"),
             ("expert_local", "rule_all"), ("expert_full", "expert_local")]
ROOT = "/content/drive/MyDrive/bitcoin_snapshots_project"
EDIR = f"{ROOT}/elliptic_plus_plus"


def load_ell(DIR=EDIR, edges=True):
    cl = pd.read_csv(f"{DIR}/txs_classes.csv")
    hdr = list(pd.read_csv(f"{DIR}/txs_features.csv", nrows=0).columns); fc = [c for c in hdr if c not in ("txId", "Time step")]
    ft = pd.read_csv(f"{DIR}/txs_features.csv", dtype={**{c: "float32" for c in fc}, "txId": "int64", "Time step": "int16"}).merge(cl, on="txId", how="left")
    ts = ft["Time step"].values.astype(int); cls = ft["class"].fillna(3).astype(int).values
    loc = [c for c in fc if c.lower().startswith("local")]; agg = [c for c in fc if c not in loc]
    print(f"{len(ft)} transactions, {len(loc)} variables locales, {len(agg)} variables agregees")
    XL = ft[loc].values.astype(np.float32); XA = ft[agg].values.astype(np.float32)
    if not edges:
        return ts, cls, XL, XA
    t2i = {int(t): i for i, t in enumerate(ft["txId"].values)}
    at = pd.read_csv(f"{DIR}/AddrTx_edgelist.txt", sep=None, engine="python"); ta = pd.read_csv(f"{DIR}/TxAddr_edgelist.txt", sep=None, engine="python")
    e = pd.concat([at.rename(columns={"input_address": "addr"})[["txId", "addr"]], ta.rename(columns={"output_address": "addr"})[["txId", "addr"]]], ignore_index=True)
    e["tx"] = e["txId"].map(t2i); e = e.dropna(subset=["tx"]); e["tx"] = e["tx"].astype(np.int64)
    codes, uniq = pd.factorize(e["addr"].astype(str))
    print(f"{len(e)} lignes d'aretes, {len(uniq)} adresses")
    return ts, cls, XL, XA, e["tx"].values, codes.astype(np.int64), len(uniq)


def history_masks(pol, t):
    if pol == "static":
        return 1, T_VA
    if pol == "win3":
        return max(1, t - 3), t - 1
    return 1, t - 1


def run_ell_blind(ts, cls, XL, XA, tx_e, addr_e, n_addr, B=500, seeds=(42, 123, 2024), out_dir=None):
    lab = np.isin(cls, [1, 2]); y = (cls == 1).astype(int)
    tr = np.where(lab & (ts <= T_TR))[0]; va = np.where(lab & (ts > T_TR) & (ts <= T_VA))[0]
    te = np.where(lab & (ts >= T_TEST0) & (ts <= T_END))[0]; yt = y[te]; tst = ts[te]
    pos_of = {int(i): k for k, i in enumerate(te)}
    R = {p: np.zeros(len(te)) for p in ELL_POL}; SEEN = {p: np.zeros(len(te), bool) for p in ELL_POL}; DOM = np.full(len(te), -1, np.int64)
    ets = ts[tx_e]; ill_tx = (cls == 1)
    for t in range(T_TEST0, T_END + 1):
        ids = te[tst == t]
        if len(ids) == 0:
            continue
        m_te = np.isin(tx_e, ids); et, ea = tx_e[m_te], addr_e[m_te]; where = np.array([pos_of[int(i)] for i in et], dtype=np.int64)
        for p in ELL_POL:
            a, b = history_masks(p, t); in_win = (ets >= a) & (ets <= b)
            tot = np.bincount(addr_e[in_win & ill_tx[tx_e]], minlength=n_addr); seen = np.bincount(addr_e[in_win], minlength=n_addr) > 0
            sc = tot[ea].astype(float)
            r = pd.Series(sc).groupby(where).max(); R[p][r.index.values] = r.values
            sn = pd.Series(seen[ea]).groupby(where).max(); SEEN[p][sn.index.values] = sn.values
            if p == "all":
                d = pd.DataFrame({"w": where, "s": sc, "a": ea}).sort_values(["w", "s"], ascending=[True, False]).drop_duplicates("w")
                DOM[d["w"].values] = np.where(d["s"].values > 0, d["a"].values, -1)
    gid = np.where(DOM >= 0, DOM, n_addr + np.arange(len(te))).astype(np.int64)
    rows = []
    for p in ELL_POL:
        S1 = R[p] > 0; S2 = (~S1) & SEEN[p]; S3 = (~S1) & (~SEEN[p])
        for name, mk in (("S1", S1), ("S2", S2), ("S3", S3), ("blind", ~S1)):
            rows.append(dict(policy=p, stratum=name, tx=int(mk.sum()), share_tx=float(mk.mean()), pos=int(yt[mk].sum()), share_pos=float(yt[mk].sum() / max(yt.sum(), 1))))
    strata = pd.DataFrame(rows); print("strates (transactions etiquetees, pas 35-49):\n", strata.round(4).to_string(index=False))
    def expert(X):
        P = np.zeros(len(te))
        for sd in seeds:
            m = fit(X[tr], y[tr], X[va], y[va], sd); P += m.predict_proba(X[te])[:, 1] / len(seeds); print(f"  graine {sd}: {m.best_iteration_} iterations")
        return np.clip(P, 0, 1 - 1e-9)
    print("expert_local (contenu)"); PL = expert(XL)
    print("expert_full (contenu + agregees)"); PF = expert(np.hstack([XL, XA]))
    S = {"expert_local": PL, "expert_full": PF}
    for p in ELL_POL:
        S[f"rule_{p}"] = R[p]; S[f"strict_{p}"] = np.where(R[p] > 0, R[p], PL)
    subs = (("pooled", np.ones(len(te), bool)), ("ts35-42", tst <= 42), ("ts43-49", tst >= 43))
    summ = pd.DataFrame([dict(model=k, test=sub, n_pos=int(yt[mk].sum()), **metrics(yt[mk], s[mk])) for k, s in S.items() for sub, mk in subs])
    print("\nmetriques:\n", summ.round(4).to_string(index=False))
    blind = pd.DataFrame([dict(policy=p, test=sub, blind_tx=int(((R[p] == 0) & mk).sum()), blind_pos=int(yt[(R[p] == 0) & mk].sum()),
                               **metrics(yt[(R[p] == 0) & mk], PL[(R[p] == 0) & mk])) for p in ELL_POL for sub, mk in subs])
    print("\nexpert_local dans chaque zone aveugle:\n", blind.round(4).to_string(index=False))
    rng = np.random.default_rng(0); diff = []
    for sub, mk in subs:
        idx = np.where(mk)[0]; yy = yt[idx]; g = np.unique(gid[idx], return_inverse=True)[1]; G = int(g.max()) + 1
        bs = {k: Boot(yy, S[k][idx]) for k in set(a for pr in ELL_PAIRS for a in pr)}; pt = {k: v.metrics(np.ones(len(idx))) for k, v in bs.items()}
        res = {pr: ([], []) for pr in ELL_PAIRS}
        for _ in range(B):
            w = np.bincount(rng.integers(0, G, G), minlength=G).astype(float)[g]; mm = {k: v.metrics(w) for k, v in bs.items()}
            for a, b in ELL_PAIRS:
                res[(a, b)][0].append(mm[a][0] - mm[b][0]); res[(a, b)][1].append(mm[a][1] - mm[b][1])
        for (a, b), (pr, ro) in res.items():
            diff.append(dict(test=sub, pair=f"{a} - {b}", dPR=pt[a][0] - pt[b][0], PR_lo=float(np.nanpercentile(pr, 2.5)), PR_hi=float(np.nanpercentile(pr, 97.5)),
                             dROC=pt[a][1] - pt[b][1], ROC_lo=float(np.nanpercentile(ro, 2.5)), ROC_hi=float(np.nanpercentile(ro, 97.5))))
    diff = pd.DataFrame(diff); print(f"\nbootstrap par grappes ({len(np.unique(gid))} groupes) :")
    for _, r in diff.iterrows():
        print(f"  {r.test:8s} {r.pair:34s} AUC-PR {r.dPR:+.4f} [{r.PR_lo:+.4f},{r.PR_hi:+.4f}]  AUC-ROC {r.dROC:+.4f} [{r.ROC_lo:+.4f},{r.ROC_hi:+.4f}]")
    if out_dir:
        for nm, d in (("strata", strata), ("summary", summ), ("blind", blind), ("diff", diff)):
            save(d, f"{out_dir}/{nm}.csv")
    return strata, summ, blind, diff


ts, cls, XL, XA, tx_e, addr_e, n_addr = load_ell()
ell_strata, ell_summ, ell_blind, ell_diff = run_ell_blind(ts, cls, XL, XA, tx_e, addr_e, n_addr, B=500, seeds=tuple(SEEDS),
                                                          out_dir=f"{ROOT}/05_results/bz/09_elliptic_blind")

In [ ]:
ts, cls, XL, XA, tx_e, addr_e, n_addr = load_ell()
ROOT = "/content/drive/MyDrive/bitcoin_snapshots_project"
ell_strata, ell_summ, ell_blind, ell_diff = run_ell_blind(ts, cls, XL, XA, tx_e, addr_e, n_addr, B=500, seeds=tuple(SEEDS),
                                                          out_dir=f"{ROOT}/05_results/bz/09_elliptic_blind")

## Expert retraining after the market shutdown (step 43)

In [ ]:
CONFIGS = [("w5", 5, 1), ("w10", 10, 1), ("all", None, 1), ("w5", 5, 4), ("w10", 10, 4), ("all", None, 4)]

def cname(n, lag):
    return f"{n}_lag{lag}"

def fit_ell(Xtr, ytr, Xva, yva, seed):
    import lightgbm as lgb
    kw = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_child_samples=20, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              reg_lambda=1.0, random_state=seed, n_jobs=-1, verbose=-1)
    if yva is not None and yva.sum() >= 3 and ytr.sum() >= 10:
        m = lgb.LGBMClassifier(n_estimators=3000, **kw)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="average_precision", callbacks=[lgb.early_stopping(150, verbose=False)])
    else:
        m = lgb.LGBMClassifier(n_estimators=100, **kw); m.fit(Xtr, ytr)
    return m

def _auc_e(y, s):
    if not 0 < y.sum() < len(y):
        return np.nan, np.nan
    return float(average_precision_score(y, s)), float(roc_auc_score(y, s))


def run_ell_refresh(ts, cls, XL, XA, B=300, seeds=(42, 123), out_dir=None):
    lab = np.isin(cls, [1, 2]); y = (cls == 1).astype(int); steps = list(range(T_TEST0, T_END + 1))
    te = np.where(lab & (ts >= T_TEST0) & (ts <= T_END))[0]; te = te[np.argsort(ts[te], kind="stable")]; yt = y[te]; tst = ts[te]
    feats = {"local": XL, "full": np.hstack([XL, XA])}; P = {}
    for fname, X in feats.items():
        tr = np.where(lab & (ts <= T_TR))[0]; va = np.where(lab & (ts > T_TR) & (ts <= T_VA))[0]; acc = np.zeros(len(te))
        for sd in seeds:
            m = fit_ell(X[tr], y[tr], X[va], y[va], sd); acc += m.predict_proba(X[te])[:, 1] / len(seeds)
        P[("fixed", fname)] = acc; print(f"{fname} fixed done")
        for n, w, lag in CONFIGS:
            out = np.zeros(len(te))
            for t in steps:
                end = t - lag; start = 1 if w is None else max(1, end - w + 1); win = lab & (ts >= start) & (ts <= end)
                vm = win & (ts >= end - 1); trm = win & (ts < end - 1); ids = tst == t
                if ids.sum() == 0:
                    continue
                pr = np.zeros(ids.sum())
                for sd in seeds:
                    m = fit_ell(X[np.where(trm)[0]], y[trm], X[np.where(vm)[0]], y[vm], sd); pr += m.predict_proba(X[te[ids]])[:, 1] / len(seeds)
                out[ids] = pr
            P[(cname(n, lag), fname)] = out; print(f"{fname} {cname(n, lag)} done")
    periods = (("35-49", np.ones(len(te), bool)), ("35-42", tst <= 42), ("43-49", tst >= 43)); rows, srows = [], []
    for (cfg, fname), p in P.items():
        for per, mk in periods:
            ap, auc = _auc_e(yt[mk], p[mk]); ms = [_auc_e(yt[mk & (tst == t)], p[mk & (tst == t)]) for t in steps if (mk & (tst == t)).any()]
            rows.append(dict(config=cfg, features=fname, period=per, n_pos=int(yt[mk].sum()), AUC_PR=ap, AUC_ROC=auc, macro_AUC_PR=float(np.nanmean([a for a, _ in ms])), macro_AUC_ROC=float(np.nanmean([b for _, b in ms]))))
        for t in steps:
            mk = tst == t; ap, auc = _auc_e(yt[mk], p[mk]); srows.append(dict(config=cfg, features=fname, step=t, n_pos=int(yt[mk].sum()), AUC_PR=ap, AUC_ROC=auc))
    met = pd.DataFrame(rows); st = pd.DataFrame(srows)
    for fname in feats:
        print(f"\nexpert {fname}, predictions poolees par periode (AUC-PR / AUC-ROC) :")
        d = met[met.features == fname]
        print(d.pivot_table(index="config", columns="period", values=["AUC_PR", "AUC_ROC", "macro_AUC_PR"], sort=False).round(4).to_string())
    rng = np.random.default_rng(0); drows = []
    for fname in feats:
        for per, mk in periods:
            idx = np.where(mk)[0]; yy = yt[idx]; pos = np.where(yy == 1)[0]; neg = np.where(yy == 0)[0]
            if len(pos) < 2:
                continue
            cfgs = [c for c, _ in P if _ == fname]; bs = {c: Boot(yy, P[(c, fname)][idx]) for c in cfgs}; pt = {c: bs[c].metrics(np.ones(len(idx))) for c in cfgs}; res = {c: ([], []) for c in cfgs if c != "fixed"}
            for _ in range(B):
                w = (np.bincount(rng.choice(pos, len(pos)), minlength=len(idx)) + np.bincount(rng.choice(neg, len(neg)), minlength=len(idx))).astype(float); b0 = bs["fixed"].metrics(w)
                for c in res:
                    mv = bs[c].metrics(w); res[c][0].append(mv[0] - b0[0]); res[c][1].append(mv[1] - b0[1])
            for c, (pr, ro) in res.items():
                drows.append(dict(features=fname, period=per, config=c, dPR=pt[c][0] - pt["fixed"][0], PR_lo=float(np.nanpercentile(pr, 2.5)), PR_hi=float(np.nanpercentile(pr, 97.5)),
                                  dROC=pt[c][1] - pt["fixed"][1], ROC_lo=float(np.nanpercentile(ro, 2.5)), ROC_hi=float(np.nanpercentile(ro, 97.5))))
    diff = pd.DataFrame(drows); print("\nconfig - fixed (point [IC 95 %]) :")
    for _, r in diff.iterrows():
        print(f"  {r.features:5s} {r.period:6s} {r.config:10s} AUC-PR {r.dPR:+.4f} [{r.PR_lo:+.4f},{r.PR_hi:+.4f}]  AUC-ROC {r.dROC:+.4f} [{r.ROC_lo:+.4f},{r.ROC_hi:+.4f}]")
    if out_dir:
        save(met, f"{out_dir}/period_metrics.csv"); save(st, f"{out_dir}/step_metrics.csv"); save(diff, f"{out_dir}/diff.csv")
    return met, st, diff


ROOT = "/content/drive/MyDrive/bitcoin_snapshots_project"
ts, cls, XL, XA = load_ell(edges=False)
ell_met, ell_st, ell_diff2 = run_ell_refresh(ts, cls, XL, XA, B=300, seeds=(42,), out_dir=f"{ROOT}/05_results/bz/10_elliptic_refresh")

In [ ]:
ts, cls, XL, XA = load_ell(edges=False)
ell_met, ell_st, ell_diff2 = run_ell_refresh(ts, cls, XL, XA, B=300, seeds=tuple(SEEDS[:3]), out_dir=f"{ROOT}/05_results/bz/10_elliptic_refresh")